In [1]:
import requests
from datetime import datetime, timedelta
from tqdm import tqdm

# Replace with your actual NASA API key
API_KEY = "J7QXvXqw9KisqQnxXOWo1Fv2DKP2NuN9Gbu7d9Na"
BASE_URL = "https://api.nasa.gov/neo/rest/v1/feed"
start_date = "2024-01-01"
end_date = "2024-01-08"

In [2]:
def get_neo_data(start_date, end_date):
    url = f"{BASE_URL}?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"
    response = requests.get(url)
    if response.status_code != 200:
        print("Failed to fetch data:", response.text)
        return None
    return response.json()

In [3]:
def extract_fields(neo_json):
    asteroids = []
    approaches = []

    for date in neo_json["near_earth_objects"]:
        for obj in neo_json["near_earth_objects"][date]:
            asteroid_data = {
                "id": int(obj["id"]),
                "name": obj["name"],
                "absolute_magnitude_h": float(obj["absolute_magnitude_h"]),
                "est_diameter_min": float(obj["estimated_diameter"]["kilometers"]["estimated_diameter_min"]),
                "est_diameter_max": float(obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"]),
                "is_hazardous": obj["is_potentially_hazardous_asteroid"]
            }

            for approach in obj["close_approach_data"]:
                approach_data = {
                    "neo_reference_id": int(obj["id"]),
                    "close_approach_date": approach["close_approach_date"],
                    "relative_velocity_kmph": float(approach["relative_velocity"]["kilometers_per_hour"]),
                    "astronomical": float(approach["miss_distance"]["astronomical"]),
                    "miss_distance_km": float(approach["miss_distance"]["kilometers"]),
                    "miss_distance_lunar": float(approach["miss_distance"]["lunar"]),
                    "orbiting_body": approach["orbiting_body"]
                }
                approaches.append(approach_data)

            asteroids.append(asteroid_data)

    return asteroids, approaches

In [ ]:
def fetch_all_data():
    start_date = datetime(2024, 1, 1)
    asteroid_data = []
    approach_data = []

    with tqdm(total=10000) as pbar:
        while len(asteroid_data) < 10000:
            end_date = start_date + timedelta(days=6)
            data = get_neo_data(start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d"))
            if not data:
                break

            asteroids, approaches = extract_fields(data)
            asteroid_data.extend(asteroids)
            approach_data.extend(approaches)

            pbar.update(len(asteroids))

            next_url = data.get("links", {}).get("next")
            if not next_url:
                break

            start_date += timedelta(days=7)

            # Avoid duplicates
            asteroid_data = list({a["id"]: a for a in asteroid_data}.values())

            if len(asteroid_data) >= 10000:
                asteroid_data = asteroid_data[:10000]
                break

    return asteroid_data, approach_data

if __name__ == "__main__":
    asteroids, approaches = fetch_all_data()

    import json
    with open("asteroids.json", "w") as f:
        json.dump(asteroids, f, indent=2)

    with open("close_approaches.json", "w") as f:
        json.dump(approaches, f, indent=2)

    print("Data saved: asteroids.json & close_approaches.json")

13158it [03:32, 61.97it/s]                          


Data saved: asteroids.json & close_approaches.json


In [5]:
import json
import pymysql

# Update your DB credentials here
DB_HOST = "localhost"
DB_USER = "root"
DB_PASSWORD = "12345"
DB_NAME = "nasa_asteroids"

def create_connection():
    return pymysql.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor
    )

def create_database(cursor):
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
    cursor.execute(f"USE {DB_NAME}")

def create_tables(cursor):
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS asteroids (
            id BIGINT,
            name VARCHAR(255),
            absolute_magnitude_h FLOAT,
            estimated_diameter_min_km FLOAT,
            estimated_diameter_max_km FLOAT,
            is_potentially_hazardous_asteroid BOOLEAN
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS close_approach (
            neo_reference_id BIGINT,
            close_approach_date DATE,
            relative_velocity_kmph FLOAT,
            astronomical FLOAT,
            miss_distance_km FLOAT,
            miss_distance_lunar FLOAT,
            orbiting_body VARCHAR(50)
        )
    """)

def insert_data(cursor, asteroids, approaches):
    asteroid_sql = """
        INSERT INTO asteroids 
        (id, name, absolute_magnitude_h, estimated_diameter_min_km, estimated_diameter_max_km, is_potentially_hazardous_asteroid)
        VALUES (%s, %s, %s, %s, %s, %s)
    """
    asteroid_values = [
        (
            a["id"], a["name"], a["absolute_magnitude_h"],
            a["est_diameter_min"], a["est_diameter_max"],
            a["is_hazardous"]
        ) for a in asteroids
    ]
    cursor.executemany(asteroid_sql, asteroid_values)

    approach_sql = """
        INSERT INTO close_approach 
        (neo_reference_id, close_approach_date, relative_velocity_kmph, astronomical, miss_distance_km, miss_distance_lunar, orbiting_body)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
    approach_values = [
        (
            a["neo_reference_id"], a["close_approach_date"], a["relative_velocity_kmph"],
            a["astronomical"], a["miss_distance_km"], a["miss_distance_lunar"],
            a["orbiting_body"]
        ) for a in approaches
    ]
    cursor.executemany(approach_sql, approach_values)

def main():
    with open("asteroids.json") as f:
        asteroids = json.load(f)

    with open("close_approaches.json") as f:
        approaches = json.load(f)

    conn = create_connection()
    with conn.cursor() as cursor:
        create_database(cursor)
        create_tables(cursor)
        insert_data(cursor, asteroids, approaches)
        conn.commit()

    print("✅ Data inserted successfully into MySQL!")

if __name__ == "__main__":
    main()

✅ Data inserted successfully into MySQL!
